### First few cells is just data preparation

My implementation is build on top of nequip and allegro python libraries to simplify data load and preprocessing.
The wigner simbols are from e3nn
Here are the links
https://github.com/mir-group/nequip
https://github.com/mir-group/allegro

In [8]:
# reference example
from nequip.data import dataset_from_config
from nequip.utils import Config
#from nequip.utils.misc import get_default_device_name
#from nequip.utils.config import _GLOBAL_ALL_ASKED_FOR_KEYS

from nequip.model import model_from_config
import os

default_config = dict(
    root="./",
    tensorboard=False,
    wandb=False,
    model_builders=[
        "SimpleIrrepsConfig",
        "EnergyModel",
        "PerSpeciesRescale",
        "StressForceOutput",
        "RescaleEnergyEtc",
    ],
    dataset_statistics_stride=1,
    device='cpu',
    default_dtype="float64",
    model_dtype="float32",
    allow_tf32=True,
    verbose="INFO",
    model_debug_mode=False,
    equivariance_test=False,
    grad_anomaly_mode=False,
    gpu_oom_offload=False,
    append=False,
    warn_unused=False,
    _jit_bailout_depth=2,  # avoid 20 iters of pain, see https://github.com/pytorch/pytorch/issues/52286
    # Quote from eelison in PyTorch slack:
    # https://pytorch.slack.com/archives/CDZD1FANA/p1644259272007529?thread_ts=1644064449.039479&cid=CDZD1FANA
    # > Right now the default behavior is to specialize twice on static shapes and then on dynamic shapes.
    # > To reduce warmup time you can do something like setFusionStrartegy({{FusionBehavior::DYNAMIC, 3}})
    # > ... Although we would wouldn't really expect to recompile a dynamic shape fusion in a model,
    # > provided broadcasting patterns remain fixed
    # We default to DYNAMIC alone because the number of edges is always dynamic,
    # even if the number of atoms is fixed:
    _jit_fusion_strategy=[("DYNAMIC", 3)],
    # Due to what appear to be ongoing bugs with nvFuser, we default to NNC (fuser1) for now:
    # TODO: still default to NNC on CPU regardless even if change this for GPU
    # TODO: default for ROCm?
    _jit_fuser="fuser1",
)
import numpy as np
import random
import torch
def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    # When running on the CuDNN backend, two further options must be set
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Set a fixed value for the hash seed
    os.environ["PYTHONHASHSEED"] = str(seed)
    #print(f"Random seed set as {seed}")

os.environ['NEQUIP_NUM_TASKS'] = '4'
# All default_config keys are valid / requested
#_GLOBAL_ALL_ASKED_FOR_KEYS.update(default_config.keys())

In [9]:
config = Config.from_file('./config/example_ETN_opt_MEA.yaml', defaults=default_config)

config['root'] = 'results/MEA_Allegro_0'
config['seed'] = 123456 + 200
#set_seed(config['seed'])
#torch.manual_seed(config['seed'])
dataset = dataset_from_config(config, prefix="dataset")

validation_dataset = None

search for AtomicData_options with prefix dataset
search for r_max with prefix dataset
          0_args :                                               r_max
instantiate TypeMapper
   optional_args :                             chemical_symbol_to_type
...TypeMapper_param = dict(
...   optional_args = {'type_names': None, 'chemical_symbol_to_type': {'Nb': 0, 'Mo': 1, 'Ta': 2, 'W': 3}, 'type_to_chemical_symbol': None, 'chemical_symbols': None},
...   positional_args = {})
instantiate register_fields
...register_fields_param = dict(
...   optional_args = {'node_fields': [], 'edge_fields': [], 'graph_fields': [], 'long_fields': []},
...   positional_args = {})
instantiate ASEDataset
   optional_args :                                            ase_args
   optional_args :                                                root
   optional_args :                                           file_name <-                                  dataset_file_name
   optional_args :                           

In [10]:
[]

[]

In [11]:
config['avg_num_neighbors'] = 26.18090057373047

In [12]:
# Trainer
from nequip.train.trainer import Trainer
from e3nn import o3

trainer = Trainer(model=None, **Config.as_dict(config))

# what is this
# to update wandb data?
config.update(trainer.params)

# = Train/test split =
trainer.set_dataset(dataset, validation_dataset)

# Some hyperparameteres
#Nc = 10 # number of chennels for F features from ETN paper
#N_rank_spec = 4 # hidden rank of reduction for type radial tensor
#config['Nc'] = Nc
#config['N_rank_spec'] = N_rank_spec

# ETN parameters
#config['d'] = 4 # dimention of the tensor train
#config['N_rank_ett'] = [4, 4, 4] # ranks of tensor train



# = Build model =
final_model = model_from_config(
    config=config, deploy=True, initialize = False, dataset=trainer.dataset_train
)

* Initialize Trainer
* Initialize Output
  ...generate file name results/MEA_Allegro_0/example/log
  ...open log file results/MEA_Allegro_0/example/log
  ...generate file name results/MEA_Allegro_0/example/metrics_epoch.csv
  ...open log file results/MEA_Allegro_0/example/metrics_epoch.csv
  ...generate file name results/MEA_Allegro_0/example/metrics_initialization.csv
  ...open log file results/MEA_Allegro_0/example/metrics_initialization.csv
  ...generate file name results/MEA_Allegro_0/example/metrics_batch_train.csv
  ...open log file results/MEA_Allegro_0/example/metrics_batch_train.csv
  ...generate file name results/MEA_Allegro_0/example/metrics_batch_val.csv
  ...open log file results/MEA_Allegro_0/example/metrics_batch_val.csv
  ...generate file name results/MEA_Allegro_0/example/best_model.pth
  ...generate file name results/MEA_Allegro_0/example/last_model.pth
  ...generate file name results/MEA_Allegro_0/example/trainer.pth
  ...generate file name results/MEA_Allegro_0/exam

...PerSpeciesScaleShift_param = dict(
...   optional_args = {'out_field': 'atomic_energy', 'scales_trainable': False, 'shifts_trainable': False, 'default_dtype': 'float32', 'num_types': 4, 'type_names': ['Nb', 'Mo', 'Ta', 'W'], 'field': 'atomic_energy', 'shifts': tensor(-11.4157), 'scales': tensor(0.8584), 'arguments_in_dataset_units': True},
...   positional_args = {'irreps_in': {'pos': 1x1oe, 'edge_index': None, 'edge_types': 1x0ee, 'node_attrs': 4x0ee, 'node_features': 4x0ee, 'edge_embedding': 8x0ee, 'edge_cutoff': 1x0ee, 'edge_attrs': 1x0ee+1x1oe+1x2ee, 'edge_features_F': 10x0ee+10x1oe+10x2ee, 'node_features_F': 10x0ee+10x1oe+10x2ee, 'node_features_ETN': 10x0ee+10x1oe+10x2ee, 'atomic_energy': 1x0ee}})
Replace string dataset_forces_rms to 0.8583642840385437
Initially outputs are globally scaled by: 0.8583642840385437, total_energy are globally shifted by None.
PerSpeciesScaleShift's arguments were in dataset units; rescaling:
  Original scales: [Nb: 0.858364, Mo: 0.858364, Ta: 0.858

In [13]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
from allegro import with_edge_spin_length
from allegro import _keys
from torch import nn
import math

#trainer.model = final_model

# Test configuration stores as dict of parameters
data0 = AtomicData.to_AtomicDataDict(dataset[1])

In [14]:
trainer.model = trainer.load_model_from_training_session('./results/MEA_Allegro_0/example')[0]

pass

instantiate TypeMapper
        all_args :                             chemical_symbol_to_type
...TypeMapper_param = dict(
...   optional_args = {'type_names': None, 'chemical_symbol_to_type': {'Mo': 1, 'Nb': 0, 'Ta': 2, 'W': 3}, 'type_to_chemical_symbol': None, 'chemical_symbols': None},
...   positional_args = {})
Building ETN model...
instantiate PairTypeEmbedding
        all_args :                                           num_types
...PairTypeEmbedding_param = dict(
...   optional_args = {'num_types': 4},
...   positional_args = {'irreps_in': None})
instantiate OneHotAtomEncoding
        all_args :                                           num_types
...OneHotAtomEncoding_param = dict(
...   optional_args = {'set_features': True, 'num_types': 4},
...   positional_args = {'irreps_in': {'pos': 1x1oe, 'edge_index': None, 'edge_types': 1x0ee}})
instantiate RadialBasisEdgeEncoding
        all_args :                                  basis_kwargs.r_max <-                                   

In [15]:
data0['pos'][0]

tensor([9.8775, 9.9013, 0.0383])

In [26]:
import torch
from torch.nn.functional import one_hot
from nequip.data import AtomicData, AtomicDataDict
from torch.nn.functional import one_hot
from e3nn.nn import FullyConnectedNet
    
from torch import nn
import math
# forward pass

trainer.model = trainer.load_model_from_training_session('./results/MEA_Allegro_0/example')[0]
data_new = trainer.model(data0)

for i in range(config['d']):
    print(trainer.model.get_submodule('model.model.func.etn.cores')[i][0, 0, 0, 0])



instantiate TypeMapper
        all_args :                             chemical_symbol_to_type
...TypeMapper_param = dict(
...   optional_args = {'type_names': None, 'chemical_symbol_to_type': {'Mo': 1, 'Nb': 0, 'Ta': 2, 'W': 3}, 'type_to_chemical_symbol': None, 'chemical_symbols': None},
...   positional_args = {})
Building ETN model...
instantiate PairTypeEmbedding
        all_args :                                           num_types
...PairTypeEmbedding_param = dict(
...   optional_args = {'num_types': 4},
...   positional_args = {'irreps_in': None})
instantiate OneHotAtomEncoding
        all_args :                                           num_types
...OneHotAtomEncoding_param = dict(
...   optional_args = {'set_features': True, 'num_types': 4},
...   positional_args = {'irreps_in': {'pos': 1x1oe, 'edge_index': None, 'edge_types': 1x0ee}})
instantiate RadialBasisEdgeEncoding
        all_args :                                  basis_kwargs.r_max <-                                   

tensor(0.1167, grad_fn=<SelectBackward0>)
tensor(0.5368, grad_fn=<SelectBackward0>)
tensor(-1.1365, grad_fn=<SelectBackward0>)
tensor(0.0348, grad_fn=<SelectBackward0>)


In [27]:
data_new['total_energy'][0]

tensor([-742.3044], grad_fn=<SelectBackward0>)

In [28]:
from allegro import rl_orthogonal, lr_orthogonal

def ortho_weights(trainer, config):
    
    # Making all non trainable
    for i in range(config['d']):
        trainer.model.get_submodule('model.model.func.etn.cores')[i].requires_grad_(False)

    for i in range(config['d']):
        trainer.model.get_parameter(f'model.model.func.etn.edge_F.{i}._forward.A').requires_grad_(False)
        trainer.model.get_parameter(f'model.model.func.etn.edge_F.{i}._forward.B').requires_grad_(False)


    # TODO: ask Max or check if sweep orthogonalization works better
    # Orthogonalization
    cores = trainer.model.get_submodule('model.model.func.etn.cores')
    instructions = []
    for i in range(config['d']):
        instructions.append([tuple(el) for el in trainer.model.get_buffer(f'model.model.func.etn.instructions_list_{i}').tolist()])
    
    ranks = [1] + trainer.model.get_buffer(f'model.model.func.etn.N_rank_ett').tolist() + [1]

    cores_new, R = rl_orthogonal(cores, ranks, instructions)

    for i in range(config['d']):
        trainer.model.get_submodule('model.model.func.etn.cores')[i] = cores_new[i]

In [29]:
ortho_weights(trainer, config)

In [30]:
for i in range(config['d']):
    print(trainer.model.get_submodule('model.model.func.etn.cores')[i][0, 0, 0, 0])

data_new = trainer.model(data0)

data_new['total_energy'][0]

tensor(-7.6804, grad_fn=<SelectBackward0>)
tensor(-0.0386, grad_fn=<SelectBackward0>)
tensor(-0.0809, grad_fn=<SelectBackward0>)
tensor(-0.0141, grad_fn=<SelectBackward0>)


tensor([-742.3044], grad_fn=<SelectBackward0>)

In [31]:
def ortho_weights(trainer, config):
    
    # Making all non trainable
    for i in range(config['d']):
        trainer.model.get_submodule('model.model.func.etn.cores')[i].requires_grad_(False)

    for i in range(config['d']):
        trainer.model.get_parameter(f'model.model.func.etn.edge_F.{i}._forward.A').requires_grad_(False)
        trainer.model.get_parameter(f'model.model.func.etn.edge_F.{i}._forward.B').requires_grad_(False)


    # TODO: ask Max or check if sweep orthogonalization works better
    # Orthogonalization
    cores = trainer.model.get_submodule('model.model.func.etn.cores')
    instructions = []
    for i in range(config['d']):
        instructions.append([tuple(el) for el in trainer.model.get_buffer(f'model.model.func.etn.instructions_list_{i}').tolist()])
    
    ranks = [1] + trainer.model.get_buffer(f'model.model.func.etn.N_rank_ett').tolist() + [1]

    cores_new, R = lr_orthogonal(cores, ranks, instructions)

    for i in range(config['d']):
        trainer.model.get_submodule('model.model.func.etn.cores')[i] = cores_new[i]

In [32]:
ortho_weights(trainer, config)

for i in range(config['d']):
    print(trainer.model.get_submodule('model.model.func.etn.cores')[i][0, 0, 0, 0])

data_new = trainer.model(data0)

data_new['total_energy'][0]

tensor(-0.0034, grad_fn=<SelectBackward0>)
tensor(-0.1309, grad_fn=<SelectBackward0>)
tensor(-0.0450, grad_fn=<SelectBackward0>)
tensor(20.0558, grad_fn=<SelectBackward0>)


tensor([-742.3044], grad_fn=<SelectBackward0>)